# AuditLens quickstart (COMPAS)

This short tutorial loads the [COMPAS](https://github.com/propublica/compas-analysis) dataset and runs **Layer 1** (`audit()` without LLM). From the repo root run `pip install -e .`, restart the kernel, then run cells top-to-bottom. Layer 2 needs `pip install -e ".[openai]"` and API keys — skipped here.

**Finding the CSV:** the next cell walks upward from `Path.cwd()` until it finds `pyproject.toml` and `src/auditlens/`, then reads `compas-scores-two-years.csv` from that root (file is gitignored; download or copy it there).

In [ ]:
from pathlib import Path
import pandas as pd
from auditlens import audit


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "auditlens").exists():
            return p
    return here


ROOT = find_repo_root()
csv_path = ROOT / "compas-scores-two-years.csv"
if not csv_path.is_file():
    raise FileNotFoundError(
        f"Expected COMPAS CSV at {csv_path}. Place compas-scores-two-years.csv in the repo root (gitignored by default)."
    )

df = pd.read_csv(csv_path)
df.head()

## 2. Run `audit()`

The last expression in the cell is `report` so Jupyter renders **`_repr_html_()`** (a table) instead of `<AuditLensReport at 0x…>`.

In [ ]:
report = audit(
    df,
    target_col="two_year_recid",
    sensitive_cols=["race", "sex"],
)
report

## 3. Summary, repr, and `to_dict()`

Use `report.summary` for severity counts, `repr(report)` for logs, and `report.to_dict()` for JSON-friendly pipelines.

In [ ]:
print(repr(report))
print(report.summary)
blob = report.to_dict()
print("to_dict keys:", sorted(blob.keys()))

## 4. Markdown export

Without Layer 2, `to_markdown()` is a Layer 1–only document. Add `task_description=...` when you are ready for LLM interpretation.

In [ ]:
print(report.to_markdown()[:1500])